# NB3 — Supply-Network Graph

**Scope.** Produces one consumer parquet:
- `outputs/nb3_seller_network_scores.parquet` — per-seller graph metrics (PageRank, in-degree, isolation flag, BFS backup, high-delay PageRank).

**Rubric surface in this notebook.** GraphFrames (vertices = sellers ∪ `customer_unique_id`; edges = purchase + serves) · PageRank · connectedComponents (GraphX backend) · motif finding `(a)-[]->(c)<-[]-(b)` · BFS · induced high-delay subgraph.

**Convergence layer** (merging this parquet with NB1 + NB2 into `seller_risk_index.parquet`) lives in `00_main.ipynb` and `src/olist/pipeline/convergence.py`. This notebook stops at the network-scores write.

**Thin-wrapper notice.** Logic in `src/olist/pipeline/network.py`. Each GraphFrame algorithm is its own `@step`-cached function so reruns on unchanged inputs skip the expensive compute.

## 1. Boot — `SparkSession` with the GraphFrames jar

In [1]:
import os, sys, inspect
from pathlib import Path

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from olist.spark_session import get_spark

spark = get_spark("nb3-supply-network", with_graphframes=True)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)


:: loading settings :: url = jar:file:/Users/lukas/Desktop/NOVA-IMS_Second_Semester/Big%20Data%20Analysis/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lukas/.ivy2/cache
The jars for the packages stored in: /Users/lukas/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-fd3fb26b-0912-480d-9ec3-6313f84503da;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 83ms :: artifacts dl 2ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	----------------------------------

26/04/24 19:47:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.8


## 2. Order-line base — delivered orders + customer_unique_id + seller_state

Uses `customer_unique_id` (not `customer_id`) per CLAUDE.md §4 so repeat-customer motifs work. `@step` caches the joined frame to `outputs/_cache/network_order_lines.parquet`.

In [2]:
from olist.pipeline.network import build_order_lines
from pyspark.sql import functions as F

order_lines = build_order_lines(spark)
print("order_lines rows:", order_lines.count())
order_lines.limit(3).show(truncate=False)


[cache] run network.order_lines (4.3s, wrote 110196 rows → outputs/_cache/network_order_lines.parquet)
order_lines rows: 110196


+--------------------------------+--------------------------------+--------------------------------+------------+-------------------+-----+
|order_id                        |customer_unique_id              |seller_id                       |seller_state|delivery_delay_days|price|
+--------------------------------+--------------------------------+--------------------------------+------------+-------------------+-----+
|55c27ec4136ec6d13577c7fb34c62b68|829d6e7a43080e1e26d649f475b9288b|cbd996ad3c1b7dc71fd0e5f5df9087e2|SP          |-17                |35.79|
|55c27ec4136ec6d13577c7fb34c62b68|829d6e7a43080e1e26d649f475b9288b|cbd996ad3c1b7dc71fd0e5f5df9087e2|SP          |-17                |35.79|
|55c27ec4136ec6d13577c7fb34c62b68|829d6e7a43080e1e26d649f475b9288b|cbd996ad3c1b7dc71fd0e5f5df9087e2|SP          |-17                |35.79|
+--------------------------------+--------------------------------+--------------------------------+------------+-------------------+-----+



## 3. Vertices — sellers ∪ unique customers

Distinct union of seller IDs and `customer_unique_id`s, each tagged with a `type` column.

In [3]:
from olist.pipeline.network import build_vertices

vertices = build_vertices(spark)
print("vertices:", vertices.count())
vertices.groupBy("type").agg(F.count("*").alias("n")).show()


[cache] run network.vertices (0.9s, wrote 96326 rows → outputs/_cache/network_vertices.parquet)
vertices: 96326
+--------+-----+
|    type|    n|
+--------+-----+
|customer|93356|
|  seller| 2970|
+--------+-----+



## 4. Edges — bidirectional `purchase` + `serves`

For each (customer_unique_id, seller_id) pair we emit one `purchase` edge (c→s) and one `serves` edge (s→c), with weight = order-item count and `avg_delay`. Bidirectionality lets PageRank flow both ways and BFS reach sellers via shared customers.

In [4]:
from olist.pipeline.network import build_edges

edges = build_edges(spark)
print("edges:", edges.count())
edges.groupBy("edge_type").agg(F.count("*").alias("n")).show()


[cache] run network.edges (0.7s, wrote 193788 rows → outputs/_cache/network_edges.parquet)
edges: 193788
+---------+-----+
|edge_type|    n|
+---------+-----+
| purchase|96894|
|   serves|96894|
+---------+-----+



## 5. GraphFrame + degree analysis

In [5]:
from olist.pipeline.network import build_graph_frame, seller_degree_stats

g = build_graph_frame(spark)
print("GraphFrame:", g)
print("\nseller_degree_stats (top 5 by purchase-only in-degree):")
seller_degrees = seller_degree_stats(spark)
seller_degrees.orderBy(F.col("in_degree_purchase_only").desc()).limit(5).show()


GraphFrame: GraphFrame(v:[id: string, type: string], e:[src: string, dst: string ... 3 more fields])

seller_degree_stats (top 5 by purchase-only in-degree):


+--------------------+------+---------------+-----------------------+
|                  id|  type|in_degree_total|in_degree_purchase_only|
+--------------------+------+---------------+-----------------------+
|6560211a19b47992c...|seller|           1790|                   1790|
|4a3ca9315b744ce9f...|seller|           1759|                   1759|
|cc419e0650a3c5ba7...|seller|           1607|                   1607|
|1f50f920176fa81da...|seller|           1383|                   1383|
|da8622b14eb17ae28...|seller|           1272|                   1272|
+--------------------+------+---------------+-----------------------+



## 6. PageRank — seller importance in the bidirectional graph

`resetProbability=0.15, maxIter=10`. Cached to `outputs/_cache/network_pagerank.parquet`.

In [6]:
from olist.pipeline.network import compute_pagerank

seller_pagerank = compute_pagerank(spark)
print("top 5 sellers by PageRank:")
seller_pagerank.orderBy(F.col("pagerank_score").desc()).limit(5).show(truncate=False)


[cache] run network.pagerank (12.4s, wrote 2970 rows → outputs/_cache/network_pagerank.parquet)
top 5 sellers by PageRank:
+--------------------------------+-----------------+
|id                              |pagerank_score   |
+--------------------------------+-----------------+
|6560211a19b47992c3666cc44a7e94c0|643.4931725240466|
|4a3ca9315b744ce9f8e9374361493884|610.6291524920101|
|cc419e0650a3c5ba77189a1882b7556a|580.410235475219 |
|1f50f920176fa81dab994f9023523100|493.8026681261715|
|955fee9216a65b617aa5c0531780ce60|458.0732470541668|
+--------------------------------+-----------------+



## 7. Connected components — flag isolated sellers

Uses the GraphX-backed algorithm; the default message-passing variant OOMs the JVM heap on ~100k vertices at 6g (see `decisions_log.md` 2026-04-22 NB3 entry).

In [7]:
from olist.pipeline.network import compute_connected_components

cc_with_size = compute_connected_components(spark)
isolated_sellers = cc_with_size.filter(
    (F.col("type") == "seller") & (F.col("component_size") == 1)
)
print(f"isolated sellers: {isolated_sellers.count():,}")
print(f"distinct components: {cc_with_size.select('component').distinct().count():,}")


26/04/24 19:48:03 WARN CacheManager: Asked to cache already cached data.


26/04/24 19:48:11 WARN BlockManager: Block rdd_604_2 already exists on this machine; not re-adding it


26/04/24 19:48:13 WARN BlockManager: Block rdd_700_1 already exists on this machine; not re-adding it


[cache] run network.connected_components (12.8s, wrote 96326 rows → outputs/_cache/network_connected_components.parquet)
isolated sellers: 0
distinct components: 1,655


## 8. Motif — shared-customer seller pairs via `(a)-[e1]->(c); (b)-[e2]->(c)`

Two sellers `a` and `b` sharing a customer `c` via two `serves` edges. Deduped with `a.id < b.id`.

In [8]:
from olist.pipeline.network import compute_shared_customer_motifs

shared_customer_pairs = compute_shared_customer_motifs(spark)
print("top shared-customer seller pairs:")
shared_customer_pairs.orderBy(F.col("n_shared_customers").desc()).limit(10).show(truncate=False)


[cache] run network.motifs_shared_customers (0.8s, wrote 3383 rows → outputs/_cache/network_motifs.parquet)
top shared-customer seller pairs:
+--------------------------------+--------------------------------+------------------+
|seller_a                        |seller_b                        |n_shared_customers|
+--------------------------------+--------------------------------+------------------+
|4a3ca9315b744ce9f8e9374361493884|cca3071e3e9bb7d12640c9fbe2301306|20                |
|1835b56ce799e6a4dc4eddc053f04066|1900267e848ceeba8fa32d80c1a5f5a8|16                |
|1025f0e2d44d7041d6cf58b6550e0bfa|1f50f920176fa81dab994f9023523100|16                |
|4a3ca9315b744ce9f8e9374361493884|d1c281d3ae149232351cd8c8cc885f0d|15                |
|391fc6631aebcf3004804e51b40bcf1e|4a3ca9315b744ce9f8e9374361493884|14                |
|88460e8ebdecbfecb5f9601833981930|f457c46070d02cadd8a68551231220dd|11                |
|4a3ca9315b744ce9f8e9374361493884|dc4a0fc896dc34b0d5bfec8438291c80|11      

## 9. BFS — backup seller for each top-10 PageRank seller

Per-seed BFS with `maxPathLength=3`; first hit other than self is the backup. Flagged `TOP10_PAGERANK_DRIVER` and `BFS_BACKUP_COLLECT` (both capped — 10 seeds, 1 row per BFS call).

In [9]:
from olist.pipeline.network import compute_bfs_backups

backup_df = compute_bfs_backups(spark)
backup_df.show(truncate=False)


[cache] run network.bfs_backups (15.9s, wrote 10 rows → outputs/_cache/network_bfs_backups.parquet)
+--------------------------------+--------------------------------+
|seller_id                       |backup_seller_id                |
+--------------------------------+--------------------------------+
|7a67c85e85bb2ce8582c35f2203ad736|c4fb51fb1c5b7c07bc5e67be6e7e8f6e|
|cc419e0650a3c5ba77189a1882b7556a|2e1c9f22be269ef4643f826c9e650a52|
|da8622b14eb17ae2831f4ac5b9dab84a|b64d51f0435e884e8de603b1655155ae|
|4a3ca9315b744ce9f8e9374361493884|6560211a19b47992c3666cc44a7e94c0|
|4869f7a5dfa277a7dca6462dcf3b52b2|ef506c96320abeedfb894c34db06f478|
|6560211a19b47992c3666cc44a7e94c0|6562efe88ce0826a4ca4f189f03b4b84|
|1f50f920176fa81dab994f9023523100|1025f0e2d44d7041d6cf58b6550e0bfa|
|955fee9216a65b617aa5c0531780ce60|b1ac6ea7895bc3dd6f0f6f4abbdd2821|
|ea8482cd71df3c1969d7b9473ff13abc|8160255418d5aaa7dbdc9f4c64ebda44|
|3d871de0142ce09b7081e2b9d1733cb1|bdb3edbaee43a761e2d4f258dc08f348|
+---------------

## 10. High-delay subgraph — re-run PageRank on late-shipping edges

Induced subgraph over edges with `avg_delay > 5`; PageRank there identifies sellers that are structurally central to *late shipments*.

In [10]:
from olist.pipeline.network import compute_delayed_subgraph_pagerank

seller_network_risk = compute_delayed_subgraph_pagerank(spark)
print("top 5 sellers by delayed-subgraph PageRank:")
seller_network_risk.orderBy(F.col("network_risk_score").desc()).limit(5).show(truncate=False)


[cache] run network.delayed_pagerank (3.8s, wrote 961 rows → outputs/_cache/network_delayed_pagerank.parquet)
top 5 sellers by delayed-subgraph PageRank:
+--------------------------------+------------------+
|id                              |network_risk_score|
+--------------------------------+------------------+
|4a3ca9315b744ce9f8e9374361493884|43.190652204318   |
|4869f7a5dfa277a7dca6462dcf3b52b2|29.413281208594544|
|1f50f920176fa81dab994f9023523100|27.199259296236534|
|7c67e1448b00f6e969d365cea6b010ab|19.70632737240356 |
|ea8482cd71df3c1969d7b9473ff13abc|19.45018260298349 |
+--------------------------------+------------------+



## 11. Assemble per-seller network scores → parquet (convergence input)

Columns: `seller_id`, `pagerank_score`, `in_degree`, `is_isolated`, `backup_seller_id`, `network_risk_score`. Written to `outputs/nb3_seller_network_scores.parquet`.

In [11]:
from olist.pipeline.network import build_seller_network_scores

seller_network_scores = build_seller_network_scores(spark)
print("seller_network_scores rows:", seller_network_scores.count())
seller_network_scores.orderBy(F.col("pagerank_score").desc()).limit(5).show(truncate=False)


[cache] run network.seller_scores (0.9s, wrote 2970 rows → outputs/nb3_seller_network_scores.parquet)
seller_network_scores rows: 2970
+--------------------------------+-----------------+---------+-----------+--------------------------------+------------------+
|seller_id                       |pagerank_score   |in_degree|is_isolated|backup_seller_id                |network_risk_score|
+--------------------------------+-----------------+---------+-----------+--------------------------------+------------------+
|6560211a19b47992c3666cc44a7e94c0|643.4931725240466|1790     |0          |6562efe88ce0826a4ca4f189f03b4b84|16.129149734446464|
|4a3ca9315b744ce9f8e9374361493884|610.6291524920101|1759     |0          |6560211a19b47992c3666cc44a7e94c0|43.190652204318   |
|cc419e0650a3c5ba77189a1882b7556a|580.410235475219 |1607     |0          |2e1c9f22be269ef4643f826c9e650a52|14.673094585384522|
|1f50f920176fa81dab994f9023523100|493.8026681261715|1383     |0          |1025f0e2d44d7041d6cf58b6550e0

## 12. Clean up

Convergence into `seller_risk_index.parquet` lives in `00_main.ipynb` — it joins the three per-seller parquets and weights them (0.35 demand + 0.35 sentiment + 0.30 network).

In [12]:
ROOT_OUT = Path.cwd().parent / "outputs" if Path.cwd().name == "notebooks" else Path.cwd() / "outputs"
print("Notebook 3 outputs:")
for path in sorted(ROOT_OUT.glob("nb3_*.parquet")):
    print(" ", path.name)
spark.stop()
print("\nSpark stopped.")


Notebook 3 outputs:
  nb3_seller_network_scores.parquet



Spark stopped.
